In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.transforms.functional as FT
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F


# Hyperparameters from the paper
LEARNING_RATE = 2e-5
DEVICE = "mps"
BATCH_SIZE = 16 # Paper used 64
WEIGHT_DECAY = 0
EPOCHS = 100
NUM_WORKERS = 2
PIN_MEMORY = True
LOAD_MODEL = False
LOAD_MODEL_FILE = "overfit.pth.tar"
IMG_DIR = "data/images"
LABEL_DIR = "data/labels"

## 1. Architecture Configuration

The architecture is defined by a list of tuples/strings to make it easy to parse dynamically.
Each tuple is: `(kernel_size, num_filters, stride, padding)`
"M" stands for MaxPool.

In [3]:
architecture_config = [
    # Tuple: (kernel_size, filters, stride, padding)
    (7, 64, 2, 3),
    "M",
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    # List: [(tuple), (tuple), repeat_times]
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 2, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
]

## 2. The CNN Block

Standard block: Conv2d -> BatchNorm -> LeakyReLU
Note: The original paper didn't use BatchNorm (it wasn't invented yet/common), but it helps convergence significantly in modern implementations.

In [4]:

class ManualCNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(ManualCNNBlock, self).__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # --- 1. CONVOLUTION PARAMETERS ---
        # Weight shape: [Out_Channels, In_Channels, K, K]
        self.conv_weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.01)
        
        # --- 2. BATCH NORM PARAMETERS ---
        self.gamma = nn.Parameter(torch.ones(1, out_channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, out_channels, 1, 1))
        
        self.eps = 1e-5

    def forward(self, x):
        # x shape: [Batch_Size (N), In_Channels (C), Height (H), Width (W)]
        N, C, H, W = x.shape
        
        # STEP 1: CONVOLUTION (via Im2Col)        
        # 1. Unfold: Extract sliding window patches
        # Output shape: [N, C*K*K, L] where L is the number of patches (H_out * W_out)
        x_unfolded = F.unfold(x, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)
        # 2. Reshape weights for Matrix Multiplication
        # We flatten the spatial dimensions of the kernel
        # Shape: [Out_Channels, C*K*K]
        weight_flat = self.conv_weight.reshape(self.out_channels, self.in_channels * self.kernel_size * self.kernel_size)
        # Calculation: [Out_Channels, C*K*K] @ [N, C*K*K, L]
        # Result shape: [N, Out_Channels, L]
        conv_out = weight_flat @ x_unfolded 
        # 4. Fold / Reshape back to image dimensions
        h_out = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        w_out = (W + 2 * self.padding - self.kernel_size) // self.stride + 1
        conv_out = conv_out.view(N, self.out_channels, h_out, w_out)
        

        # STEP 2: BATCH NORMALIZATION
        # Calculate Mean and Variance across Batch (N) and Spatial (H, W) dimensions
        mean = conv_out.mean(dim=(0, 2, 3), keepdim=True)
        var = conv_out.var(dim=(0, 2, 3), keepdim=True, unbiased=True)
        x_norm = (conv_out - mean) / torch.sqrt(var + self.eps)
        # Scale and Shift (Gamma and Beta)
        bn_out = self.gamma * x_norm + self.beta

        # STEP 3: LEAKY RELU        
        # Logic: if x > 0 return x, else return 0.1 * x
        out = torch.where(bn_out > 0, bn_out, bn_out * 0.1)

        return out


x = torch.randn(2, 3, 32, 32)
model = ManualCNNBlock(in_channels=3, out_channels=16)
output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 3, 32, 32])
Output shape: torch.Size([2, 16, 32, 32])


## 3. The YOLO Model

This class parses the `architecture_config` list and creates the PyTorch layers.
It ends with the Fully Connected layers that output the `7x7x30` tensor.

In [10]:
class Yolo(nn.Module):
    def __init__(self, in_channels=3, **kwargs):
        super().__init__()
        self.architecture = architecture_config
        self.in_channels = in_channels
        self.backbone = self._create_conv_layers(self.architecture)
        self.head = self._create_fcs(**kwargs)

    def forward(self, x):
        x = self.backbone(x)
        out = self.head(torch.flatten(x, start_dim=1))
        return out

    def _create_conv_layers(self, architecture):
        layers = []
        in_channels = self.in_channels

        for x in architecture:
            if type(x) == tuple:
                layers += [
                    ManualCNNBlock(
                        in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3]
                    )
                ]
                in_channels = x[1]

            elif x == "M":
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]

            elif type(x) == list:
                conv1 = x[0]
                conv2 = x[1]
                num_repeats = x[2]

                for _ in range(num_repeats):
                    layers += [
                        ManualCNNBlock(
                            in_channels,
                            conv1[1],
                            kernel_size=conv1[0],
                            stride=conv1[2],
                            padding=conv1[3],
                        )
                    ]
                    layers += [
                        ManualCNNBlock(
                            conv1[1],
                            conv2[1],
                            kernel_size=conv2[0],
                            stride=conv2[2],
                            padding=conv2[3],
                        )
                    ]
                    in_channels = conv2[1]

        return nn.Sequential(*layers)

    def _create_fcs(self, split_size, num_boxes, num_classes):
        S, B, C = split_size, num_boxes, num_classes

        # The dense layers described in the paper
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.Dropout(0.5),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, S * S * (C + B * 5)),
        )

## 4. The Loss Function

The famous YOLO Loss consisting of 5 parts:
1. Box Coordinate Loss (center x, y)
2. Box Size Loss (width, height)
3. Object Confidence Loss (is there an object?)
4. No-Object Confidence Loss (is there NO object?)
5. Classification Loss (what object is it?)

Dimensions needed: (Batch, 7, 7, 30)

In [11]:
def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Calculates intersection over union
    
    Parameters:
        boxes_preds (tensor): Predictions of Bounding Boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): Correct labels of Bounding Boxes (BATCH_SIZE, 4)
        box_format (str): midpoint/corners, if boxes (x,y,w,h) or (x1,y1,x2,y2)
    
    Returns:
        tensor: Intersection over union for all examples
    """

    if box_format == "midpoint":
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # .clamp(0) is for the case when they do not intersect
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    return intersection / (box1_area + box2_area - intersection + 1e-6)

class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.mse = nn.MSELoss(reduction="sum")
        self.S = S
        self.B = B
        self.C = C
        self.lambda_noobj = 0.5
        self.lambda_coord = 5

    def forward(self, predictions, target):
        # predictions are shaped (BATCH_SIZE, S*S(C+B*5)) -> (N, 1470)
        # target is shaped (BATCH_SIZE, S, S, 25) (C + 1 + 4)
        predictions = predictions.reshape(-1, self.S, self.S, self.C + self.B * 5)

        # Calculate IoU for the two predicted bounding boxes with target bbox
        # predictions[..., 21:25] is box 1 (x, y, w, h)
        # predictions[..., 26:30] is box 2 (x, y, w, h)
        iou_b1 = intersection_over_union(predictions[..., 21:25], target[..., 21:25])
        iou_b2 = intersection_over_union(predictions[..., 26:30], target[..., 21:25])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)
        
        # Take the box with highest IoU out of the two prediction
        iou_maxes, bestbox = torch.max(ious, dim=0)
        # bestbox indicates which box (0 or 1) is responsible for the object
        exists_box = target[..., 20].unsqueeze(3) # Iobj_i

        # ======================= #
        #   FOR BOX COORDINATES   #
        # ======================= #
        # Set boxes contains the coordinates of the chosen box for each cell
        # We check both box 1 and box 2. If bestbox is 0, we take box 1, else box 2.
        box_predictions = exists_box * (
            (
                bestbox * predictions[..., 26:30]
                + (1 - bestbox) * predictions[..., 21:25]
            )
        )

        box_targets = exists_box * target[..., 21:25]

        # Take sqrt of width, height of boxes
        # We use torch.sign to preserve sign (though w,h should be >0) and abs + 1e-6 for stability
        box_predictions[..., 2:4] = torch.sign(box_predictions[..., 2:4]) * torch.sqrt(
            torch.abs(box_predictions[..., 2:4] + 1e-6)
        )
        box_targets[..., 2:4] = torch.sqrt(box_targets[..., 2:4])

        # (N, S, S, 4) -> (N*S*S, 4)
        box_loss = self.mse(
            torch.flatten(box_predictions, end_dim=-2),
            torch.flatten(box_targets, end_dim=-2),
        )

        # ==================== #
        #   FOR OBJECT LOSS    #
        # ==================== #
        # pred_box is the confidence score for the box with highest IoU
        pred_box = (
            bestbox * predictions[..., 25:26] + (1 - bestbox) * predictions[..., 20:21]
        )
        # exists_box is the ground truth confidence (1 if object exists)
        object_loss = self.mse(
            torch.flatten(exists_box * pred_box),
            torch.flatten(exists_box * target[..., 20:21]),
        )

        # ======================= #
        #   FOR NO OBJECT LOSS    #
        # ======================= #
        # (N, S, S, 1) -> (N, -1)
        
        # Loss for box 1 (index 20)
        no_object_loss = self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 20:21], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1),
        )
        
        # Loss for box 2 (index 25)
        no_object_loss += self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 25:26], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1)
        )

        # ================== #
        #   FOR CLASS LOSS   #
        # ================== #
        # (N, S, S, 20) -> (N*S*S, 20)
        class_loss = self.mse(
            torch.flatten(exists_box * predictions[..., :20], end_dim=-2),
            torch.flatten(exists_box * target[..., :20], end_dim=-2),
        )

        loss = (
            self.lambda_coord * box_loss
            + object_loss
            + self.lambda_noobj * no_object_loss
            + class_loss
        )

        return loss

## 5. Training Loop Skeleton

In [12]:
# Imports needed for training
import os
from PIL import Image
import matplotlib.pyplot as plt

# Hyperparameters
LEARNING_RATE = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
WEIGHT_DECAY = 0
EPOCHS = 10
NUM_WORKERS = 2
PIN_MEMORY = True
IMG_DIR = "images"
LABEL_DIR = "labels"

class WIDERFaceDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir, S=7, B=2, C=20, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.S = S
        self.B = B
        self.C = C
        self.label_files = []
        
        for root, dirs, files in os.walk(label_dir):
            for file in files:
                if file.endswith(".txt"):
                    self.label_files.append(os.path.join(root, file))
        
        print(f"Found {len(self.label_files)} label files.")

    def __len__(self):
        return len(self.label_files)

    def __getitem__(self, index):
        label_path = self.label_files[index]
        rel_path = os.path.relpath(label_path, self.label_dir)
        img_path = os.path.join(self.img_dir, rel_path.replace(".txt", ".jpg"))
        
        boxes = []
        try:
            with open(label_path) as f:
                for line in f.readlines():
                    parts = line.replace("\n", "").split()
                    if not parts: continue
                    class_label, x, y, width, height = [
                        float(x) if float(x) != int(float(x)) else int(x)
                        for x in parts
                    ]
                    boxes.append([class_label, x, y, width, height])
        except Exception as e:
            print(f"Error reading {label_path}: {e}")

        try:
             image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error reading image {img_path}: {e}")
            image = Image.new('RGB', (448, 448))

        boxes = torch.tensor(boxes)

        if self.transform:
            image = self.transform(image)
        else:
            image = image.resize((448, 448))
            image = transforms.ToTensor()(image)

        label_matrix = torch.zeros((self.S, self.S, self.C + 5))
        
        if boxes.nelement() != 0:
            for box in boxes:
                class_label, x, y, width, height = box.tolist()
                class_label = int(class_label)
                j, i = int(self.S * x), int(self.S * y)
                if j >= self.S: j = self.S - 1
                if i >= self.S: i = self.S - 1
                x_cell = self.S * x - j
                y_cell = self.S * y - i
                width_cell = width * self.S
                height_cell = height * self.S
                
                if label_matrix[i, j, 20] == 0:
                    label_matrix[i, j, 20] = 1 
                    label_matrix[i, j, 21:25] = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    if class_label < self.C:
                        label_matrix[i, j, class_label] = 1

        return image, label_matrix

def train_fn(train_loader, model, optimizer, loss_fn):
    loop = tqdm(train_loader, leave=True)
    mean_loss = []

    for batch_idx, (x, y) in enumerate(loop):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = loss_fn(out, y)
        mean_loss.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())
        
    return sum(mean_loss)/len(mean_loss)

model = Yolo(split_size=7, num_boxes=2, num_classes=20).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = YoloLoss()

print("Setting up dataset...")
train_dataset = WIDERFaceDataset(
    img_dir=IMG_DIR,
    label_dir=LABEL_DIR,
    transform=None,
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0, # Using 0 to avoid potential issues in notebook environment
    pin_memory=PIN_MEMORY,
    shuffle=True,
    drop_last=True,
)

print(f"Starting training for {EPOCHS} epochs...")
loss_history = []

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    avg_loss = train_fn(train_loader, model, optimizer, loss_fn)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1} loss: {avg_loss}")

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()


Setting up dataset...
Found 3226 label files.
Starting training for 10 epochs...

Epoch [1/10]


  0%|          | 0/201 [00:03<?, ?it/s]


KeyboardInterrupt: 